# Floor 3's score of record (s3-genret `referee full`)

**The founder runs this, in his own Colab.** It produces `results.json`, the score of record for a GR claim. Not yet
run in Colab (written 2026-11-13, CPU development only this round). The first run is the check of prediction
G0-steward-2.

Before running it:
- the claim's predictions are committed in the lab's `lab/predictions/` (the researcher's job);
- the held-out draw exists in your Drive (`lab/held-out/genret-esci.md`);
- you know the claim's commit and the command that writes its run file and sid map (from the predictions).

The runtime is an L4 by default (`colab-gpu-run` skill). The baselines on ESCI-US build s3-search's engine index on
1.22M products.

In [ ]:
# 1. A clean checkout of s3-genret at the claim's commit (the repository is public: no token).
GENRET_REF = "main"   # set this to the claim's full commit SHA (a short SHA cannot be fetched)
import os, subprocess
subprocess.run(["git", "clone", "--quiet", "https://github.com/hanialshater/s3-genret", "/content/s3-genret"], check=True)
os.chdir("/content/s3-genret")
subprocess.run(["git", "checkout", "--quiet", GENRET_REF], check=True)
!git log -1 --format='%H %s'
!pip -q install -r referee/requirements.txt

In [ ]:
# 2. The pinned s3-search checkout, ESCI and esci-sample; then the fast mode (must say ok).
!python -m referee setup
!python -m referee fast

In [ ]:
# 3. The held-out draws, from your Drive. GENRET_HELD_OUT is required for a score of record.
from google.colab import drive
drive.mount("/content/drive")
os.environ["GENRET_HELD_OUT"] = "/content/drive/MyDrive/shakabuku-held-out/genret-esci-v1.parquet"
sf = "/content/drive/MyDrive/shakabuku-held-out/search-esci-v1.parquet"   # the search floor's, if drawn
if os.path.exists(sf):
    os.environ["S3_SEARCH_HELD_OUT"] = sf
print({k: os.path.exists(v) for k, v in os.environ.items() if k.endswith("HELD_OUT")})

In [ ]:
# 4. The search floor's baselines on every ESCI-US test query (k 1000), by s3-search's code at the pin.
!pip -q install -r .cache/s3-search/requirements.txt
!python -m referee baselines --dataset=esci-us --k=1000 --device=cuda --out-dir=/content/baselines-esci-us

In [ ]:
# 5. The claim's run file and sid map: run the command from the claim's predictions here, from this checkout.
RUN = "/content/run.jsonl.gz"        # written by the claim's command
SID_MAP = "/content/sid_map.parquet"  # written by the claim's command
# !python -m <the researcher's package> ... --out=$RUN --sid-map-out=$SID_MAP

In [ ]:
# 6. The score of record.
!python -m referee full --run={RUN} --sid-map={SID_MAP} --baselines-dir=/content/baselines-esci-us --out=/content/results.json
import shutil, time
dst = f"/content/drive/MyDrive/shakabuku-results/genret-{time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())}"
os.makedirs(dst, exist_ok=True)
for f in ("/content/results.json", "/content/baselines-esci-us/baselines.json"):
    shutil.copy(f, dst)
print(dst)

Afterwards: stop the runtime. Put `results.json` and `baselines.json` in the lab's records, with the claim's verdict
against its predictions. Never commit the held-out draw.